# Traces and pads

The copper of a circuit board is built from paths offset to traces, round pads and drilled holes, with every edge
named after its net. The example has three nets on a pitch of 2.54 mm, each a path with a 45° jog. A trace is the
path offset by half its width of 0.4 on both sides, with round ends.

In [1]:
import math
import planex as px

In [2]:
from docs_figures import Row, Vertices

In [3]:
routes = {}
for k in range(3):
    y0, y1 = 2.54 * k, 2.54 * (k + 1)
    routes[f"net{k}"] = [(0, y0), (5, y0), (7.54, y1), (12, y1)]

traces = {net: px.FromPolyline(path, name=net).Offset(0.2) for net, path in routes.items()}

## Check

A trace of half-width d = 0.2 covers 2d along its length L, plus two round ends of together π d². At each of the two
45° bends, a sector of d² θ / 2 is added on the outside. On the inside, the term of the length counts a kite of
d² tan(θ / 2) twice, so it is subtracted once.

In [4]:
d, theta = 0.2, math.pi / 4
L = 5 + 2.54 * math.sqrt(2) + (12 - 7.54)
exact = 2 * d * L + math.pi * d**2 + 2 * d**2 * (theta / 2 - math.tan(theta / 2))
for net, trace in traces.items():
    print(net, "area", trace.mass, "exact", exact)

net0 area 5.344783527060708 exact 5.344783527060708
net1 area 5.34478352706071 exact 5.344783527060708
net2 area 5.344783527060707 exact 5.344783527060708


## Pads and drills

Round pads of diameter 1.7 at both ends of each trace are united with the traces, and drills of diameter 1.0 are
subtracted (Figure 1). The pads take the name of their net, and the drills are named `drill`.

In [5]:
copper = list(traces.values())
drills = []
for net, path in routes.items():
    for x, y in (path[0], path[-1]):
        copper.append(px.QuickSketch().Circle(x, y, 0.85, name=net).Face())
        drills.append(px.QuickSketch().Circle(x, y, 0.5, name="drill").Face())

board = px.Subtract([px.Unify(copper), *drills])
print(sorted({e.name for e in board.edges}), len(board.wires), "loops")

['drill', 'net0', 'net1', 'net2'] 9 loops


In [6]:
Row((px.Compound(list(traces.values())), "Three traces"), (board, "The board"),
    caption="Three nets: traces, then pads united and drills subtracted")